# v2.0b: Follow-Up Tests on the Walk-Forward Backtest

Where is Track B v2.0 weakest? Uses the backtest files already made (no new fits):
`trackbtv_w730_<season>_scored.csv` (v2.0), `trackbxg_w365_<season>_scored.csv` (v1.1) and
`baselines_<season>.csv` (rolling xG, home rate).

**Sections**
1. Setup
2. Results by subset: early season, rest / back-to-backs, unexpected starters, favorites
3. Cost of guessing the wrong starting goalie
4. Home-win share: predicted vs actual
5. Summary: what each finding points to in v2

Lower log loss is better. "bias" = mean predicted P(home win) minus actual home-win share (negative = home team
won more than predicted). Subsets are small, so every difference comes with a 90% bootstrap range.

## 1. Setup

In [ ]:
%load_ext autoreload
%autoreload 2
%matplotlib inline

import sys; sys.path.insert(0, "..")
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from config.settings import OUTPUT_DIR
from src.data.db import get_connection
from src.validation.scoring import log_loss, calibration_slope, paired_difference

def check(label, ok, note=""):
    print(f"{'PASS ' if ok else 'CHECK'}  {label}" + (f"   ({note})" if note else ""))

BT = OUTPUT_DIR / "backtests"
SEASONS = ["20242025", "20252026"]
FILES = {"v2.0": "trackbtv_w730", "v1.1": "trackbxg_w365",     # labels of the backtests compared
         "v2.0d rest": "trackbtv_w730_r", "v2.0d rest+home": "trackbtv_w730_rh"}  # used if present

COL = {"v1.1": "v11", "v2.0d rest": "v20d_r", "v2.0d rest+home": "v20d_rh"}
frames = []
for season in SEASONS:
    main = BT / f"{FILES['v2.0']}_{season}_scored.csv"
    if not main.exists():
        print(f"missing {main.name}"); continue
    d = pd.read_csv(main, parse_dates=["game_date"]).rename(columns={"p_home": "p_v20"})
    for name, lab in FILES.items():
        if name == "v2.0":
            continue
        f = BT / f"{lab}_{season}_scored.csv"
        if f.exists():
            d = d.merge(pd.read_csv(f, usecols=["game_id", "p_home"]).rename(columns={"p_home": "p_" + COL[name]}),
                        on="game_id", how="left")
    b = BT / f"baselines_{season}.csv"
    if b.exists():
        d = d.merge(pd.read_csv(b, usecols=["game_id", "p_home_rate", "p_rolling_xg"]), on="game_id", how="left")
    frames.append(d)
df = pd.concat(frames, ignore_index=True).sort_values(["game_date", "game_id"]).reset_index(drop=True)
df["season"] = df["season"].astype(str)
MODELS = {"v2.0": "p_v20", "v2.0d rest": "p_v20d_r", "v2.0d rest+home": "p_v20d_rh", "v1.1": "p_v11",
          "rolling xG": "p_rolling_xg", "home rate": "p_home_rate"}
MODELS = {k: v for k, v in MODELS.items() if v in df and df[v].notna().all()}
print(f"{len(df):,} games; models: {', '.join(MODELS)}")

## 2. Results by subset

For each subset: games, log loss per model, v2.0 minus rolling xG (paired, 90% range), and v2.0's bias.

In [ ]:
# Per-team context from the schedule of these games (each file holds the full regular season)
long = pd.concat([df[["game_id", "game_date", "season", "home"]].rename(columns={"home": "team"}).assign(side="home"),
                  df[["game_id", "game_date", "season", "away"]].rename(columns={"away": "team"}).assign(side="away")])
long = long.sort_values(["team", "game_date", "game_id"])
long["games_before"] = long.groupby(["season", "team"]).cumcount()
long["rest_days"] = long.groupby(["season", "team"])["game_date"].diff().dt.days
ctx = long.pivot_table(index="game_id", columns="side", values=["games_before", "rest_days"], aggfunc="first")
ctx.columns = [f"{a}_{b}" for a, b in ctx.columns]
df = df.drop(columns=[c for c in ctx.columns if c in df]).merge(ctx.reset_index(), on="game_id", how="left")

def subset_table(groups: dict) -> pd.DataFrame:
    rows = []
    for name, mask in groups.items():
        sub = df[mask]
        if len(sub) < 30:
            continue
        row = {"subset": name, "games": len(sub)}
        for m, col in MODELS.items():
            row[m] = log_loss(sub[col], sub["home_won"])
        if "rolling xG" in MODELS:
            r = paired_difference(sub, MODELS["v2.0"], MODELS["rolling xG"], n_boot=1000)
            row["v2.0 - rolling xG"] = f"{r['difference']:+.4f} [{r['lo']:+.4f}, {r['hi']:+.4f}]"
        for m in [m for m in MODELS if m.startswith("v2.0")]:
            row[f"{m} bias"] = sub[MODELS[m]].mean() - sub["home_won"].mean()
        rows.append(row)
    return pd.DataFrame(rows).round(4)

everything = {"all games": df["game_id"].notna()}

### 2a. Early season

Ratings are carried over the summer (one off-season step, loosely identified). `games before` = the fewer of the
two teams' games already played this season.

In [ ]:
gb = df[["games_before_home", "games_before_away"]].min(axis=1)
early = subset_table({**everything, "first 10 games": gb < 10, "games 10-29": (gb >= 10) & (gb < 30),
                      "game 30+": gb >= 30})
display(early)
first = df[gb < 10]
a, b = calibration_slope(first[MODELS["v2.0"]], first["home_won"])
print(f"v2.0 calibration slope in each team's first 10 games: {b:.2f} (1 = calibrated, >1 too cautious)")

### 2b. Rest and back-to-backs

The model has no fatigue term. If tired teams lose more than predicted, the bias shows it: an away team on the
second night of a back-to-back should make the home team win *more* than predicted (negative bias).

In [ ]:
h1 = df["rest_days_home"] == 1
a1 = df["rest_days_away"] == 1
rest = subset_table({**everything,
                     "home on back-to-back only": h1 & ~a1,
                     "away on back-to-back only": a1 & ~h1,
                     "both on back-to-back": h1 & a1,
                     "neither": ~h1 & ~a1,
                     "home 3+ days rest, away back-to-back": (df["rest_days_home"] >= 3) & a1})
display(rest)
for name, mask in [("home on back-to-back only", h1 & ~a1), ("away on back-to-back only", a1 & ~h1)]:
    s = df[mask]
    if len(s) >= 30:
        se = np.sqrt(s["home_won"].var() / len(s))
        preds = ", ".join(f"{m} {s[MODELS[m]].mean():.1%}" for m in MODELS if m.startswith("v2.0"))
        print(f"{name}: actual home win {s['home_won'].mean():.1%} (± {1.645 * se:.1%} at 90%); predicted: {preds}")

### 2c. Unexpected starting goalies

The backtest used the actual starters. "Unexpected" = the starter was not the team's most frequent starter in its
last 10 games (what `predict_day` would have guessed). These are mostly backup starts.

In [ ]:
from src.models.game_model import default_starter
conn = get_connection()
guess = {}
for r in df[["game_id", "game_date", "home", "away"]].itertuples():
    day = str(r.game_date.date())
    guess[(r.game_id, "home")] = default_starter(conn, r.home, day)
    guess[(r.game_id, "away")] = default_starter(conn, r.away, day)
df["guess_home"] = [guess[(g, "home")] for g in df["game_id"]]
df["guess_away"] = [guess[(g, "away")] for g in df["game_id"]]
df["surprise_home"] = df["guess_home"].notna() & (df["guess_home"] != df["home_goalie"])
df["surprise_away"] = df["guess_away"].notna() & (df["guess_away"] != df["away_goalie"])
print(f"Default guess wrong: home {df.surprise_home.mean():.1%}, away {df.surprise_away.mean():.1%} of team-games")
goalies = subset_table({**everything,
                        "no surprise starter": ~df.surprise_home & ~df.surprise_away,
                        "home surprise starter": df.surprise_home & ~df.surprise_away,
                        "away surprise starter": df.surprise_away & ~df.surprise_home,
                        "both surprise": df.surprise_home & df.surprise_away})
display(goalies)

### 2d. Favorites vs coin flips

Grouped by how strongly v2.0 favored one side. Bias here is for the favorite: predicted minus actual share of
favorite wins (negative = favorites won more than predicted, i.e. the model was too cautious).

In [ ]:
p = df[MODELS["v2.0"]]
fav = np.maximum(p, 1 - p)
fav_won = np.where(p >= 0.5, df["home_won"], 1 - df["home_won"])
bins = pd.cut(fav, [0.5, 0.55, 0.6, 0.65, 1.0], labels=["50-55%", "55-60%", "60-65%", "65%+"], include_lowest=True)
t = pd.DataFrame({"bin": bins, "fav": fav, "won": fav_won}).groupby("bin", observed=True).agg(
    games=("won", "size"), predicted=("fav", "mean"), actual=("won", "mean"))
t["band_90"] = 1.645 * np.sqrt(t.predicted * (1 - t.predicted) / t.games)
t["favorite bias"] = t.predicted - t.actual
t["inside"] = t["favorite bias"].abs() <= t.band_90
display(t.round(3))
check("favorites win about as often as predicted in every group", t["inside"].all())

## 3. Cost of guessing the wrong starting goalie

Live predictions guess the starter; the backtest used the real one. Each game is recomputed with the guessed
starters by swapping goalie offsets in the saved scoring rates (no refit):
lam_home' = lam_home * exp(gamma_away_actual - gamma_away_guess), and the same for the away side.
The change in P(home win) from the exact game model (plug-in, posterior-mean rates) is added to the logged prediction.

In [ ]:
from src.processing.goalie import goalie_game_xg
from src.processing.game_table import goalie_gamma
from src.processing.late_game import load_params
from src.models.game_model import outcome_probs

gg = goalie_game_xg(conn)
gam = goalie_gamma(gg, sorted(df["game_date"].dt.normalize().unique()))
gam["game_date"] = pd.to_datetime(gam["game_date"])
look = gam.set_index(["game_date", "goalie_id"])["gamma"]
def offset(day, gid):
    if pd.isna(gid):
        return 0.0
    return float(look.get((day, int(gid)), 0.0))
day = df["game_date"].dt.normalize()
df["gamma_home_guess"] = [offset(d, g) for d, g in zip(day, df["guess_home"])]
df["gamma_away_guess"] = [offset(d, g) for d, g in zip(day, df["guess_away"])]

late = load_params(OUTPUT_DIR / "models" / "late_game.json")
RHO = 0.52                                       # tie inflation (v1.1 and v2.0 both estimate about 0.52)
lh, la = df["lam_home"].to_numpy(), df["lam_away"].to_numpy()
lh_g = lh * np.exp(df["gamma_away"] - df["gamma_away_guess"]).to_numpy()
la_g = la * np.exp(df["gamma_home"] - df["gamma_home_guess"]).to_numpy()
p_act = outcome_probs(lh, la, np.full(len(df), RHO), late)["home_win"]
p_gss = outcome_probs(lh_g, la_g, np.full(len(df), RHO), late)["home_win"]
df["p_v20_guess"] = np.clip(df[MODELS["v2.0"]] + (p_gss - p_act), 0.01, 0.99)

wrong = df.surprise_home | df.surprise_away
r = paired_difference(df, "p_v20_guess", MODELS["v2.0"])
print(f"Games with at least one wrong guess: {wrong.mean():.1%} ({wrong.sum():,} of {len(df):,})")
print(f"Mean change in P(home win) when the guess is wrong: {np.abs(df.p_v20_guess - df[MODELS['v2.0']])[wrong].mean() * 100:.1f} pts")
print(f"Log loss, actual starters {log_loss(df[MODELS['v2.0']], df.home_won):.4f} vs guessed {log_loss(df.p_v20_guess, df.home_won):.4f}")
print(f"Guessed minus actual, all games: {r['difference']:+.4f} [{r['lo']:+.4f}, {r['hi']:+.4f}]")
w = df[wrong]
r2 = paired_difference(w, "p_v20_guess", MODELS["v2.0"])
print(f"Guessed minus actual, games with a wrong guess: {r2['difference']:+.4f} [{r2['lo']:+.4f}, {r2['hi']:+.4f}]")
print("\nRead: the all-games number is what live predictions lose to starter guessing; compare it with the "
      "v2.0 vs v1.1 gain (0.0025) to see whether waiting for confirmed starters matters.")

## 4. Home-win share: predicted vs actual

In-sample (Phase 6) the model was about 1 point low (53.4% vs 54.2%). If it is low out of sample too, home ice
(one league-wide value) may need to vary by season.

In [ ]:
hs = df.groupby("season").agg(games=("home_won", "size"), actual=("home_won", "mean"),
                              **{m: (c, "mean") for m, c in MODELS.items()})
hs.loc["both"] = [len(df), df.home_won.mean()] + [df[c].mean() for c in MODELS.values()]
hs["v2.0 - actual"] = hs["v2.0"] - hs["actual"]
hs["90% band"] = 1.645 * np.sqrt(hs["actual"] * (1 - hs["actual"]) / hs["games"])
display(hs.round(4))
for s, row in hs.iterrows():
    check(f"{s}: predicted home-win share within the 90% band", abs(row["v2.0 - actual"]) <= row["90% band"],
          f"{row['v2.0']:.1%} vs {row['actual']:.1%}")
by_m = df.assign(month=df.game_date.dt.to_period("M").astype(str)).groupby("month").agg(
    predicted=(MODELS["v2.0"], "mean"), actual=("home_won", "mean"), games=("home_won", "size"))
ax = by_m[["predicted", "actual"]].plot(marker="o", figsize=(10, 3.5), title="Home-win share by month (v2.0)")
ax.set_ylabel("share"); plt.show()

## 5. Summary

In [ ]:
lines = []
e = early.set_index("subset")
if "first 10 games" in e.index and "rolling xG" in MODELS:
    lines.append(f"Early season (first 10 games): v2.0 {e.loc['first 10 games', 'v2.0']:.4f} vs rolling xG "
                 f"{e.loc['first 10 games', 'rolling xG']:.4f}; v2.0 - rolling xG {e.loc['first 10 games', 'v2.0 - rolling xG']}")
rr = rest.set_index("subset")
for k in ["home on back-to-back only", "away on back-to-back only"]:
    if k in rr.index:
        lines.append(f"{k}: v2.0 bias {rr.loc[k, 'v2.0 bias']:+.3f} (negative = home won more than predicted)")
lines.append(f"Wrong starter guess in {wrong.mean():.1%} of games; costs {r['difference']:+.4f} log loss overall")
lines.append(f"Home-win share: predicted {df[MODELS['v2.0']].mean():.1%} vs actual {df.home_won.mean():.1%}")
print("\n".join(lines))